In [1]:
# # 02. Бечмарк без ML та Перевірка Воріт Якості (Quality Gate 1)
# **Проєкт:** E-Commerce Customer Retention (`ecommerce_retention_lab1`)
# **Мета:** Розрахунок детермінованого бейзлайну, демонстрація спрацювання `validate_full_schema` / `guard_config_integrity`, збирання та перевірка адаптивного конвеєра препроцесингу.

# %%
import sys
from pathlib import Path

BASE_DIR = Path("..").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import pytest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from core.dataset_loader import (
    extract_features_and_target,
    get_prepared_dataset,
    read_data_config,
)
from core.pipeline__factory import (
    PipelineSchemaComposer,
    guard_config_integrity,
    validate_full_schema,
)

In [2]:
# ## 1. Розрахунок Non-ML Heuristic Benchmark
# **Правило:** `Tenure <= 1` **АБО** `Complain == 1` -> Клієнт піде у відтік (`Churn = 1`).

# %%
data_config = read_data_config("config/dataset_config.yaml")
df = get_prepared_dataset(data_config, apply_injection=True)
X, y_true = extract_features_and_target(df, data_config)

# Застосування евристичного правила
y_heuristic = ((X["Tenure"] <= 1) | (X["Complain"] == 1)).astype(int)

acc = accuracy_score(y_true, y_heuristic)
rec = recall_score(y_true, y_heuristic, pos_label=1)
prec = precision_score(y_true, y_heuristic, pos_label=1)
f1 = f1_score(y_true, y_heuristic, pos_label=1)

print("=" * 50)
print("РЕЗУЛЬТАТИ ЕВРИСТИЧНОГО БЕЙЗЛАЙНУ (NON-ML BENCHMARK)")
print("=" * 50)
print(f"Accuracy:  {acc:.4f}")
print(f"Recall:    {rec:.4f} (Ціль ML: >= 0.80)")
print(f"Precision: {prec:.4f} (Ціль ML: >= 0.50)")
print(f"F1-Score:  {f1:.4f}")
print("=" * 50)

РЕЗУЛЬТАТИ ЕВРИСТИЧНОГО БЕЙЗЛАЙНУ (NON-ML BENCHMARK)
Accuracy:  0.6988
Recall:    0.8481 (Ціль ML: >= 0.80)
Precision: 0.3413 (Ціль ML: >= 0.50)
F1-Score:  0.4867


In [3]:
# ## 2. Демонстрація спрацювання Воріт Якості (Quality Gate)
# Перевірка блокування спроби включити захищені атрибути (`CityTier` або `Gender`) у секцію `features`.

# %%
bad_config = {
    "model_family": "scale_sensitive",
    "features": {
        "numeric_continuous": ["Tenure"],
        "categorical_ordinal": ["CityTier"],  # ПОМИЛКА: Захищена ознака у features!
    },
    "protected_attributes": ["CityTier", "Gender"],
    "imputation_policy": {
        "numeric": [{"max_share": 1.0, "strategy": "median"}],
        "categorical": [{"max_share": 1.0, "strategy": "most_frequent"}],
    },
    "transform_policy": {
        "scale_sensitive": {
            "numeric_continuous": ["standard_scaler"],
            "categorical_ordinal": ["ordinal_encoder"],
        }
    },
}

print("Запуск перевірки зіпсованої конфігурації...")
try:
    guard_config_integrity(bad_config)
except ValueError as e:
    print("\n[УСПІШНО ПЕРЕХОПЛЕНО ПОМИЛКУ ВХІДНИХ ВОРІТ ЯКОСТІ]:")
    print(e)

Запуск перевірки зіпсованої конфігурації...

[УСПІШНО ПЕРЕХОПЛЕНО ПОМИЛКУ ВХІДНИХ ВОРІТ ЯКОСТІ]:
[Quality Gate Violation] Захищені ознаки потрапили в перелік features: {'CityTier'}


In [4]:
# ## 3. Збирання та тренування адаптивного ColumnTransformer

# %% [markdown]
# ### 3.1. Стратегія А: `scale_sensitive` (Linear / Additive models)

# %%
composer_linear = PipelineSchemaComposer("config/pipeline_linear.yaml")
ct_linear = composer_linear.assemble()

# Навчання на тренувальній вибірці
ct_linear.fit(X, y_true)
print("ColumnTransformer (scale_sensitive) успішно навчено!")

audit_df_linear = composer_linear.extract_audit_summary(ct_linear)
print("\nАудиторський слід імпутації (scale_sensitive):")
print(audit_df_linear.head(10))

ColumnTransformer (scale_sensitive) успішно навчено!

Аудиторський слід імпутації (scale_sensitive):
                       feature               group  observed_missing_share  \
0                       Tenure  numeric_continuous                  0.0469   
1              WarehouseToHome  numeric_continuous                  0.0446   
2               HourSpendOnApp  numeric_continuous                  0.0453   
3     NumberOfDeviceRegistered  numeric_continuous                  0.0000   
4              NumberOfAddress  numeric_continuous                  0.0000   
5  OrderAmountHikeFromlastYear  numeric_continuous                  0.0471   
6                   CouponUsed  numeric_continuous                  0.0455   
7                   OrderCount  numeric_continuous                  0.0458   
8            DaySinceLastOrder  numeric_continuous                  0.0545   
9               CashbackAmount  numeric_continuous                  0.2000   

   applied_threshold chosen_strategy imp

In [5]:
# ### 3.2. Стратегія Б: `scale_invariant` (Tree-based models)

# %%
composer_tree = PipelineSchemaComposer("config/pipeline_tree.yaml")
ct_tree = composer_tree.assemble()

ct_tree.fit(X, y_true)
print("ColumnTransformer (scale_invariant) успішно навчено!")

audit_df_tree = composer_tree.extract_audit_summary(ct_tree)
print("\nАудиторський слід імпутації (scale_invariant):")
print(audit_df_tree.head(10))

ColumnTransformer (scale_invariant) успішно навчено!

Аудиторський слід імпутації (scale_invariant):
                       feature               group  observed_missing_share  \
0                       Tenure  numeric_continuous                  0.0469   
1              WarehouseToHome  numeric_continuous                  0.0446   
2               HourSpendOnApp  numeric_continuous                  0.0453   
3     NumberOfDeviceRegistered  numeric_continuous                  0.0000   
4              NumberOfAddress  numeric_continuous                  0.0000   
5  OrderAmountHikeFromlastYear  numeric_continuous                  0.0471   
6                   CouponUsed  numeric_continuous                  0.0455   
7                   OrderCount  numeric_continuous                  0.0458   
8            DaySinceLastOrder  numeric_continuous                  0.0545   
9               CashbackAmount  numeric_continuous                  0.2000   

   applied_threshold chosen_strategy imp

In [6]:
# ## 4. Експорт підсумкової таблиці метаданих ознак

# %%
meta_table = audit_df_linear[
    ["feature", "group", "observed_missing_share", "chosen_strategy"]
].copy()
meta_table.rename(
    columns={
        "observed_missing_share": "missing_share",
        "chosen_strategy": "rule",
    },
    inplace=True,
)

summary_df = composer_linear.config
policy_summary = composer_linear.extract_audit_summary(ct_linear)

output_path = Path("reports/feature_meta_summary.md")
if not output_path.is_absolute():
    from core.pipeline_factory import BASE_DIR
    output_path = BASE_DIR / output_path

output_path.parent.mkdir(parents=True, exist_ok=True)

# Формування Markdown-звіту
markdown_output = meta_table.to_markdown(index=False)
output_path.write_text(markdown_output, encoding="utf-8")
print(f"Звіт метаданих збережено у {output_path}")

Звіт метаданих збережено у C:\Users\Emin\PycharmProject\PML\reports\feature_meta_summary.md


C:\Users\Emin\AppData\Local\Temp\ipykernel_8916\3125814883.py:7: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  meta_table.rename(
